In [1]:
# First time: pip install google-genai numpy sounddevice python-dotenv (or use your env manager).
# If you already have a 16kHz mono PCM16 WAV, set USE_MIC=False and point TEST_CASES to your files.
import io
import os
import wave
from datetime import datetime
from pathlib import Path

import numpy as np
import sounddevice as sd
from dotenv import load_dotenv
from IPython.display import Audio, display
from google import genai
from google.genai import types

load_dotenv()

GEMINI_API_KEY = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")

MODEL_ID = "gemini-2.5-flash-native-audio-preview-12-2025"
VOICE_NAME = "Kore"  # See supported voices in the Live API docs.
INPUT_SAMPLE_RATE = 16000
OUTPUT_SAMPLE_RATE = 24000

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Toggle for mic vs. prerecorded WAV input.
USE_MIC = True
MIC_SECONDS = 5.0
MIC_INPUT_WAV = OUTPUT_DIR / f"mic_input_{RUN_TAG}.wav"

# Test cases: edit these to change instruction or voice per run.
TEST_CASES = [
    {
        "id": "neutral_1",
        "wav": "outputs/gemini_input.wav",
        "instruction": "Talk like Yoda.",
        "voice": VOICE_NAME,
    },
]


## Gemini Live (Speech-to-Speech)

Reference: https://ai.google.dev/gemini-api/docs/live-guide
Voice list: https://docs.cloud.google.com/vertex-ai/generative-ai/docs/live-api/configure-language-voice#voices_supported

Notes:
- Input audio is raw 16-bit PCM, 16kHz mono (output is 24kHz PCM).
- Configure tests in the first code cell (`TEST_CASES`, `USE_MIC`, `OUTPUT_DIR`).
- Outputs are saved in `notebooks/outputs/` with a timestamp and a JSON manifest. This folder is gitignored.


In [2]:
def read_pcm16_wav(path: str, sample_rate: int = INPUT_SAMPLE_RATE) -> bytes:
    with wave.open(path, "rb") as wf:
        channels = wf.getnchannels()
        sampwidth = wf.getsampwidth()
        rate = wf.getframerate()
        if channels != 1 or sampwidth != 2 or rate != sample_rate:
            raise ValueError(
                f"Expected mono PCM16 {sample_rate}Hz WAV. Got channels={channels}, "
                f"sampwidth={sampwidth}, rate={rate}."
            )
        return wf.readframes(wf.getnframes())


def pcm16_to_wav_bytes(pcm: bytes, sample_rate: int = OUTPUT_SAMPLE_RATE, channels: int = 1) -> bytes:
    buf = io.BytesIO()
    with wave.open(buf, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(2)
        wf.setframerate(sample_rate)
        wf.writeframes(pcm)
    return buf.getvalue()


# Record mic audio and save a 16kHz mono PCM16 WAV for the API.
def record_wav(
    seconds: float = 5.0,
    sample_rate: int = INPUT_SAMPLE_RATE,
    output_path: str | None = None,
):
    output_path = output_path or str(MIC_INPUT_WAV)
    print(f"Recording {seconds}s at {sample_rate}Hz...")
    # If this hangs, make sure the kernel has microphone permission.
    audio = sd.rec(int(seconds * sample_rate), samplerate=sample_rate, channels=1, dtype="float32")
    sd.wait()
    pcm = (np.clip(audio, -1, 1) * 32767).astype(np.int16)

    path = Path(output_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with wave.open(str(path), "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sample_rate)
        wf.writeframes(pcm.tobytes())

    print(f"Saved {output_path}")


In [3]:
# Send the input WAV to the Gemini Live API and return audio + transcripts.
async def gemini_live_s2s(
    input_wav_path: str,
    output_wav_path: str | None = None,
    model: str = MODEL_ID,
    voice_name: str = VOICE_NAME,
    instruction_text: str | None = None,
):
    if not GEMINI_API_KEY:
        raise ValueError("Missing GEMINI_API_KEY or GOOGLE_API_KEY in notebooks/.env")

    # Client picks up GEMINI_API_KEY from env if not passed.
    client = genai.Client(api_key=GEMINI_API_KEY)

    pcm_in = read_pcm16_wav(input_wav_path, sample_rate=INPUT_SAMPLE_RATE)

    config = {
        "response_modalities": ["AUDIO"],
        "input_audio_transcription": {},
        "output_audio_transcription": {},
    }
    if voice_name:
        config["speech_config"] = {
            "voice_config": {"prebuilt_voice_config": {"voice_name": voice_name}}
        }

    def _merge(prev: str | None, new: str) -> str:
        if not prev:
            return new
        if new.startswith(prev):
            return new
        if prev.startswith(new):
            return prev
        return prev + " " + new

    async with client.aio.live.connect(model=model, config=config) as session:
        if instruction_text:
            try:
                await session.send_realtime_input(text=instruction_text)
            except TypeError:
                print(
                    "Warning: this SDK version does not support text realtime input. "
                    "Speak the instruction in the audio or update google-genai."
                )

        await session.send_realtime_input(
            audio=types.Blob(
                data=pcm_in,
                mime_type=f"audio/pcm;rate={INPUT_SAMPLE_RATE}",
            )
        )
        # If the stream pauses and you don't get a response, flush with:
        # await session.send_realtime_input(audio_stream_end=True)

        out_pcm = bytearray()
        input_final = None
        output_final = None
        input_partial = None
        output_partial = None

        async for response in session.receive():
            # Some SDK builds surface audio bytes here.
            if response.data is not None:
                out_pcm.extend(response.data)
            # Others surface audio in inline_data parts.
            elif response.server_content and response.server_content.model_turn:
                for part in response.server_content.model_turn.parts:
                    if part.inline_data and isinstance(part.inline_data.data, (bytes, bytearray)):
                        out_pcm.extend(part.inline_data.data)

            # Transcription events are independent and can arrive out of order.
            in_tx = None
            out_tx = None
            if hasattr(response, "input_transcription") and response.input_transcription:
                in_tx = response.input_transcription
            if hasattr(response, "output_transcription") and response.output_transcription:
                out_tx = response.output_transcription
            if response.server_content:
                in_tx = in_tx or response.server_content.input_transcription
                out_tx = out_tx or response.server_content.output_transcription

            if in_tx and in_tx.text is not None:
                if getattr(in_tx, "finished", False):
                    input_final = _merge(input_final, in_tx.text)
                    input_partial = None
                else:
                    # Some SDKs send deltas; others send full partials.
                    input_partial = _merge(input_partial, in_tx.text)

            if out_tx and out_tx.text is not None:
                if getattr(out_tx, "finished", False):
                    output_final = _merge(output_final, out_tx.text)
                    output_partial = None
                else:
                    # Some SDKs send deltas; others send full partials.
                    output_partial = _merge(output_partial, out_tx.text)

            if response.server_content and response.server_content.turn_complete and (output_final or output_partial):
                break

    wav_bytes = pcm16_to_wav_bytes(bytes(out_pcm), sample_rate=OUTPUT_SAMPLE_RATE)
    if output_wav_path:
        path = Path(output_wav_path)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(wav_bytes)

    transcripts = {
        "input": input_final or input_partial,
        "output": output_final or output_partial,
    }
    return wav_bytes, transcripts


In [4]:
import json


async def run_cases():
    if USE_MIC:
        record_wav(seconds=MIC_SECONDS, output_path=str(MIC_INPUT_WAV))

    results = []
    for case in TEST_CASES:
        case_id = case.get("id")
        if not case_id:
            raise ValueError("Each test case needs an 'id'.")

        input_wav = str(MIC_INPUT_WAV) if USE_MIC else case.get("wav")
        if not input_wav:
            raise ValueError(f"Missing 'wav' for case {case_id} (or set USE_MIC=True).")

        instruction = case.get("instruction")
        voice = case.get("voice", VOICE_NAME)
        out_wav = OUTPUT_DIR / f"{case_id}_{RUN_TAG}_gemini.wav"

        audio_bytes, transcripts = await gemini_live_s2s(
            input_wav_path=str(input_wav),
            output_wav_path=str(out_wav),
            model=MODEL_ID,
            voice_name=voice,
            instruction_text=instruction,
        )

        manifest = {
            "case_id": case_id,
            "timestamp": datetime.now().isoformat(),
            "provider": "gemini",
            "model": MODEL_ID,
            "voice": voice,
            "instruction": instruction,
            "input_wav": str(input_wav),
            "output_wav": str(out_wav),
            "transcripts": transcripts,
        }
        manifest_path = OUTPUT_DIR / f"{case_id}_{RUN_TAG}_gemini.json"
        manifest_path.write_text(json.dumps(manifest, indent=2))

        print(f"{case_id}: wrote {out_wav} and {manifest_path}")
        results.append(
            {
                "case_id": case_id,
                "audio_bytes": audio_bytes,
                "transcripts": transcripts,
                "manifest_path": str(manifest_path),
                "output_wav": str(out_wav),
            }
        )

    return results


results = await run_cases()


Recording 5.0s at 16000Hz...
Saved outputs/mic_input_20260204_143150.wav


neutral_1: wrote outputs/neutral_1_20260204_143150_gemini.wav and outputs/neutral_1_20260204_143150_gemini.json


In [5]:
# Play the last generated audio.
if results:
    display(Audio(results[-1]["audio_bytes"]))
